# Use Case 6: Spotting a Fake Product Listing (E-commerce)

**The real-world problem:** online marketplaces (Amazon, eBay, Flipkart...) have millions of sellers uploading listings every day. Two everyday problems:

1. **Messy categories** — a seller uploads a product, but which shelf/department does it actually belong on?
2. **Counterfeits** — some sellers list fake/knockoff goods pretending to be the real brand.

This notebook uses no new TypeSafe ideas at all — just `Choice` (notebook 1) and `Noul` (notebook 3), asked about the same listing in one request, exactly like the checklist pattern from notebook 4. No loops this time, nice and simple, one listing at a time.

### 🏪 Analogy

Imagine a flea market inspector walking past a stall. For every item on the table, they do two quick things in their head:

1. **"Which department does this belong in?"** — Footwear? Watches? Electronics? (drops it into a mailbox, exactly like notebook 1)
2. **"Does something about this smell fake?"** — words like "replica", "same as original but cheaper", missing box/receipt (pushes a Yes/No dial, exactly like notebook 3)

The inspector does both checks **at the same time**, just by glancing at the item once — they don't need two separate trips. That's exactly what we'll do: one request, two questions.

## Step 1 — Install the SDK

In [1]:
%pip install typesafe-sdk

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Add your API key

In [ ]:
import os

# 👉 PASTE YOUR TYPESAFE API KEY BELOW, between the quotes
os.environ["TYPESAFE_API_KEY"] = "PASTE_YOUR_API_KEY_HERE"

print("Key saved for this session.")

## Step 3 — Create the client

We import `Choice` and `Noul` together — one request, two question types, same as notebook 4's guardrail mix.

In [3]:
from typesafe_sdk import Choice, Noul, NoulCriteria, TypeSafeClient

client = TypeSafeClient()
print("Client is ready.")

Client is ready.


## Step 4 — The listing we want to check (the "state")

Same pattern as every notebook so far: `state` is just the text TypeSafe should read. Here it's a seller's product listing (title + description, combined into one block of text).

In [ ]:
listing_text = (
    "Rolex-style wristwatch, AAA mirror quality replica, looks identical to the "
    "original for a fraction of the price. No box or papers included."
)

state = {"document": listing_text}

## Step 5 — Define our two questions

- **`category`** — a `Choice`, exactly like notebook 1's ticket sorter, just with product departments instead of support topics.
- **`is_counterfeit_risk`** — a `Noul`, exactly like notebook 3's yes/no dial, with `criteria` spelling out what "looks fake" actually means (same reason we added it in notebook 5 — this is a judgment call, not an obvious keyword match).

In [ ]:
questions = {
    "category": Choice(
        instructions="Which department does this product listing belong in?",
        criteria={
            "footwear": "Shoes, sneakers, boots",
            "watches": "Wristwatches and watch accessories",
            "electronics": "Phones, computers, gadgets",
            "other": "Anything that doesn't fit the categories above",
        },
    ),
    "is_counterfeit_risk": Noul(
        instructions="Does this listing show signs of being a counterfeit or fake product, rather than a genuine authorized item?",
        criteria=NoulCriteria(
            true="The listing uses words like 'replica', 'AAA quality', 'inspired by', or 'same as original but cheaper', or otherwise signals it isn't the genuine branded product.",
            false="Nothing suggests this is anything other than a genuine, authorized product.",
        ),
    ),
}

## Step 6 — Send it to TypeSafe

One request, both questions answered together — same `client.system_one(...)` call you've used in every notebook.

In [ ]:
response = client.system_one(
    state=state,
    questions=questions,
)

response

## Step 7 — Read the answers

Nothing new: `.choices[...]` for the `Choice` question, `.nouls[...]` for the `Noul` question — same accessors as notebooks 1 and 3.

In [ ]:
category = response.choices["category"].choice
counterfeit_score = response.nouls["is_counterfeit_risk"].noul

print("Listing        :", listing_text)
print("Category       :", category)
print("Counterfeit risk:", counterfeit_score)

You should see `category = "watches"` and a **high** `counterfeit_score` (close to 1.0) — the words "replica" and "AAA mirror quality" are textbook fake-listing language.

## Step 8 — Turn the number into a decision

Same idea as notebook 4's guardrail: TypeSafe gives you a number, *you* decide what counts as risky enough to act on.

In [ ]:
if counterfeit_score > 0.7:
    print("🚩 Flag this listing for manual review before it goes live.")
else:
    print("✅ No counterfeit red flags — approve the listing.")

## Step 9 — Try it yourself: a genuine listing

Same two questions, reused on a completely different, legitimate listing. Watch both the category **and** the counterfeit score change.

In [ ]:
# ✏️ Edit this text, then re-run this cell (Shift+Enter)
listing_text_2 = (
    "Official Nike Air Max sneakers, brand new in box, includes original receipt "
    "and authenticity card. Purchased directly from an authorized Nike retailer."
)

response_2 = client.system_one(
    state={"document": listing_text_2},
    questions=questions,  # reusing the same two questions from Step 5
)

category_2 = response_2.choices["category"].choice
counterfeit_score_2 = response_2.nouls["is_counterfeit_risk"].noul

print("Listing        :", listing_text_2)
print("Category       :", category_2)
print("Counterfeit risk:", counterfeit_score_2)

if counterfeit_score_2 > 0.7:
    print("🚩 Flag this listing for manual review before it goes live.")
else:
    print("✅ No counterfeit red flags — approve the listing.")

This one should land on `category = "footwear"` with a **low** counterfeit score — "official", "authorized retailer", and "authenticity card" are exactly the opposite signal from "replica".

## Step 10 — Clean up

In [ ]:
client.close()
print("Closed.")

## Recap — what did we just learn?

1. "Normalizing listings" (sorting into the right category) is just `Choice` — nothing new.
2. "Detecting counterfeits" is just `Noul` with `criteria` spelling out what counts as a red flag — nothing new.
3. Both questions get asked **in the same request** about the same listing, by putting them both in one `questions` dictionary — same checklist trick from notebook 4.
4. Deciding what to do with the numbers (`🚩 flag` vs `✅ approve`) is plain Python with a threshold — again, same as notebook 4.

The pattern really doesn't change use case to use case: **describe the text (`state`) → ask one or more structured questions (`Choice`/`Score`/`Noul`) → read the numbers back → decide what to do with plain Python.** That's the whole toolkit.

👉 Tell me which use case to build next whenever you're ready.